# Лабораторная работа № 3. Почасовое включение генераторов

Вариант 11 · Google Colab · **Среда выполнения → Выполнить всё**.

In [ ]:
%pip install -q numpy==2.3.5 scipy==1.17.0 matplotlib==3.10.8

## Исходные данные

Мощности заданы шаблоном. Тарифы, стоимость пуска и повторяющийся профиль спроса имеют небольшой случайный разброс. Базовый расчёт использует первые 12 часов. Используем отдельный `data.json`. Если файла нет, создаём его с `seed=10012`.

In [ ]:
import csv
import json
import os
from contextlib import redirect_stderr, redirect_stdout
from ctypes import CDLL
from pathlib import Path
from itertools import product
from time import perf_counter
from io import BytesIO

import numpy as np
from IPython.display import JSON, Markdown, display
from IPython.display import Image
from scipy.optimize import Bounds, LinearConstraint, linprog, milp
from scipy.sparse import lil_matrix

SEED = 10012
TOL = 1e-7
STATUS = {0: "optimal", 1: "limit", 2: "infeasible", 3: "unbounded", 4: "solver_error"}

LAB_DIR = Path.cwd()
if LAB_DIR.name != "lr3":
    LAB_DIR = LAB_DIR / "lr3"
DATA_PATH = LAB_DIR / "data.json"
RESULTS = LAB_DIR / "results"


def generators(rng):
    # Масштабирование по префиксам одного суточного профиля.
    profile = np.array([43, 58, 87, 106, 137, 94, 51, 54, 90, 127, 85, 49])
    return {
        "seed": SEED, "ids": ["G1", "G2", "G3", "G4"],
        "minimum": [20, 12, 9, 5], "maximum": [62, 43, 36, 24],
        "variable_cost": ([1500, 2020, 2670, 3610] + rng.integers(-25, 26, 4)).tolist(),
        "startup_cost": ([90000, 54000, 30000, 10000] + rng.integers(-500, 501, 4)).tolist(),
        "demand": (np.tile(profile, 16) + rng.integers(-3, 4, 192)).tolist(),
    }


if not DATA_PATH.is_file():
    LAB_DIR.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(np.random.SeedSequence([SEED, 3]))
    DATA_PATH.write_text(
        json.dumps(generators(rng), ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
data = json.loads(DATA_PATH.read_text(encoding="utf-8"))

## Решатель и сохранение результатов

In [ ]:
def write_csv(folder, name, headers, rows):
    path = folder / f"{name}.csv"
    path.parent.mkdir(exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(headers)
        writer.writerows(rows)


def write_check(folder, name, result):
    path = folder / f"{name}.json"
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def solve_lp(cost, *, label, verbose=True, **kwargs):
    print(f"\n=== {label} ===", flush=True)
    result = linprog(
        cost, method="highs-ds", options={"presolve": True, "disp": verbose}, **kwargs
    )
    print(f"{STATUS.get(result.status, result.status)} | {result.message}", flush=True)
    return result


def require_status(result, *allowed):
    if result.status not in allowed:
        raise RuntimeError(result.message)


def run_with_log(run, data, folder):
    folder.mkdir(parents=True, exist_ok=True)
    with (folder / "solver.log").open("w", encoding="utf-8", buffering=1) as log:
        # HiGHS пишет лог напрямую в stdout, минуя Python.
        stdout_fd = os.dup(1)
        try:
            os.dup2(log.fileno(), 1)
            with redirect_stdout(log), redirect_stderr(log):
                return run(data)
        finally:
            CDLL(None).fflush(None)
            os.dup2(stdout_fd, 1)
            os.close(stdout_fd)


def show_table(path):
    with path.open(encoding="utf-8", newline="") as file:
        header, *rows = csv.reader(file)
    table = [header, ["---"] * len(header), *rows]
    display(Markdown("\n".join("| " + " | ".join(row) + " |" for row in table)))

## Построение модели

In [ ]:
def inputs(data, hours):
    minimum = np.array(data["minimum"])
    available = np.broadcast_to(data["maximum"], (hours, len(minimum))).copy()
    demand = np.array(data["demand"][:hours])
    return minimum, available, demand, np.array(data["variable_cost"]), np.array(data["startup_cost"])


def build_model(minimum, available, demand, energy_cost, start_cost, multiplier=1):
    hours, count = available.shape
    size = hours * count
    p, y, v = np.arange(3 * size).reshape(3, hours, count)
    balance = lil_matrix((hours, 3 * size))
    rows = lil_matrix((5 * size, 3 * size))
    right = np.zeros(5 * size)
    for t in range(hours):
        balance[t, p[t]] = 1
        for g in range(count):
            row = (t * count + g) * 5
            rows[row, p[t, g]] = 1
            rows[row, y[t, g]] = -multiplier * available[t, g]
            rows[row + 1, p[t, g]] = -1
            rows[row + 1, y[t, g]] = minimum[g]
            rows[row + 2, y[t, g]] = 1
            rows[row + 2, v[t, g]] = -1
            rows[row + 3, v[t, g]] = 1
            rows[row + 3, y[t, g]] = -1
            rows[row + 4, v[t, g]] = 1
            if t > 0:
                rows[row + 2, y[t - 1, g]] = -1
                rows[row + 4, y[t - 1, g]] = 1
            right[row + 4] = 1
    return {
        "cost": np.r_[np.tile(energy_cost, hours), np.zeros(size), np.tile(start_cost, hours)],
        "binary": np.r_[np.zeros(size), np.ones(2 * size)],
        "upper": np.r_[available.ravel(), np.ones(2 * size)],
        "A_eq": balance.tocsr(), "b_eq": demand,
        "A_ub": rows.tocsr(), "b_ub": right,
    }


def solve_milp(model, label, *, time_limit=None, verbose=True):
    options = {"presolve": True, "mip_rel_gap": 0.0, "disp": verbose}
    if time_limit is not None:
        options["time_limit"] = time_limit
    bounds = Bounds(0, model["upper"])
    constraints = [LinearConstraint(model["A_eq"], model["b_eq"], model["b_eq"]),
                   LinearConstraint(model["A_ub"], -np.inf, model["b_ub"])]
    if verbose:
        print(f"\n=== {label} ===", flush=True)
    start = perf_counter()
    result = milp(model["cost"], integrality=model["binary"], bounds=bounds,
                  constraints=constraints, options=options)
    elapsed = perf_counter() - start
    print(f"{label}: {STATUS.get(result.status, result.status)} | {result.message} | {elapsed:.6f} s",
          flush=True)
    return result, elapsed


def relaxation(model, label):
    return solve_lp(model["cost"], label=label, A_ub=model["A_ub"], b_ub=model["b_ub"],
                    A_eq=model["A_eq"], b_eq=model["b_eq"],
                    bounds=np.column_stack((np.zeros(len(model["cost"])), model["upper"])))

## Независимая проверка и дополнительные расчёты

In [ ]:
def max_violation(values):
    return float(np.max(values, initial=0))


def verify_objective(checks, expected, actual):
    checks["objective_recomputed"] = float(actual)
    checks["objective_error"] = float(abs(expected - actual))
    if any(value > TOL for key, value in checks.items() if key != "objective_recomputed"):
        raise ValueError(f"Independent check failed: {checks}")
    return checks


def cost_of(plan, energy_cost, start_cost):
    p, _, v = plan
    return np.sum(p * energy_cost) + np.sum(v * start_cost)


def verify(plan, minimum, available, demand, energy_cost, start_cost, objective,
           *, multiplier=1, integral=True):
    p, y, v = plan
    previous = np.vstack((np.zeros(y.shape[1]), y[:-1]))
    checks = {
        "demand_balance": max_violation(abs(p.sum(axis=1) - demand)),
        "physical_upper": max_violation(p - available),
        "minimum_output": max_violation(minimum * y - p),
        "big_m_upper": max_violation(p - multiplier * available * y),
        "binary_box": max(max_violation(-y), max_violation(y - 1),
                          max_violation(-v), max_violation(v - 1)),
        "output_lower": max_violation(-p),
        "start_lower": max_violation(y - previous - v),
        "start_upper_on": max_violation(v - y),
        "start_upper_previous": max_violation(v + previous - 1),
    }
    if integral:
        checks["integrality"] = max(max_violation(abs(y - np.rint(y))),
                                    max_violation(abs(v - np.rint(v))))
        checks["startup_exactness"] = max_violation(abs(v - np.maximum(0, y - previous)))
    return verify_objective(checks, objective, cost_of(plan, energy_cost, start_cost))


def greedy(minimum, available, demand, energy_cost, start_cost):
    count = len(minimum)
    states = np.array(list(product([0, 1], repeat=count)))
    order = np.argsort(energy_cost)
    plan = np.zeros((3, len(demand), count))
    previous = np.zeros(count)
    for t, load in enumerate(demand):
        best = np.inf
        for state in states:
            p = minimum * state
            maximum = available[t] * state
            if not (p.sum() <= load <= maximum.sum()):
                continue
            remaining = load - p.sum()
            for g in order:
                extra = min(remaining, maximum[g] - p[g])
                p[g] += extra
                remaining -= extra
            starts = np.maximum(0, state - previous)
            value = energy_cost @ p + start_cost @ starts
            if value < best:
                best = value
                plan[:, t] = p, state, starts
        if not np.isfinite(best):
            raise ValueError(f"No feasible hourly configuration at hour {t + 1}")
        previous = plan[1, t]
    return plan


def benchmark(data):
    samples, summary = [], []
    for hours in [12, 24, 48, 96, 192]:
        args = inputs(data, hours)
        model = build_model(*args)
        timings, gaps, statuses = [], [], []
        for repeat in range(1, 4):
            result, elapsed = solve_milp(model, f"scale {hours} h #{repeat}",
                                         time_limit=0.4, verbose=(repeat == 1))
            require_status(result, 0, 1)
            gap = None
            if result.x is not None:
                verify(result.x.reshape(3, *args[1].shape), *args, result.fun)
                if result.mip_gap is not None:
                    gap = 100 * result.mip_gap
                    gaps.append(gap)
            timings.append(elapsed)
            statuses.append(STATUS[result.status])
            samples.append([hours, model["cost"].size, int(model["binary"].sum()),
                            repeat, statuses[-1], elapsed, gap, result.fun, result.mip_dual_bound,
                            result.mip_node_count])
        summary.append([hours, model["cost"].size, int(model["binary"].sum()),
                        np.median(timings), np.median(gaps) if gaps else None,
                        statuses.count("optimal")])
    write_csv(RESULTS, "scaling_runs", ["hours", "variables", "binaries", "repeat", "status",
                                     "seconds", "gap_pct", "objective", "dual_bound", "nodes"], samples)
    write_csv(RESULTS, "scaling", ["hours", "variables", "binaries", "median_seconds",
                                "median_gap_pct", "optimal_runs"], summary)
    return np.array(summary, dtype=float)


def draw_benchmark(summary):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter

    plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10})
    fig, axes = plt.subplots(1, 2, figsize=(6.35, 2.65), layout="constrained")
    hours = summary[:, 0].astype(int)
    for ax, metric, label in zip(axes, [summary[:, 3], summary[:, 4]],
                                 ["Медиана времени, с", "Медиана зазора, %"]):
        ax.plot(hours, metric, "o-", color="#212121", linewidth=1.3, markersize=4, clip_on=False)
        ax.set_xscale("log", base=2)
        ax.set_xticks(hours, [str(value) for value in hours])
        ax.tick_params(axis="x", which="minor", bottom=False, labelbottom=False)
        ax.set(xlabel="Горизонт, ч", ylabel=label, ylim=(0, None))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.2f}".replace(".", ",")))
        ax.grid(axis="y", color="#d9d9d9", linewidth=0.65)
        for edge in ["top", "right"]:
            ax.spines[edge].set_visible(False)
    fig.savefig(RESULTS / "scaling.pdf")
    plt.close(fig)
    return fig

## Эксперименты

Три сценария MILP, LP-релаксация, завышенное $M$, жадный план и измерения при увеличении горизонта.

In [ ]:
def run_lab(data):
    minimum, capacity, load, energy_cost, start_cost = inputs(data, 12)
    reduced = capacity.copy()
    reduced[3:7, 1] -= 8
    stressed = reduced.copy()
    stressed[4:6, 2] -= 10
    peak = load.copy()
    peak[[4, 9]] += 7
    runs = []
    checks = {}
    plan_rows = []
    for name, available, demand in [("base", capacity, load), ("deficit", reduced, load),
                                    ("stress", stressed, peak)]:
        args = minimum, available, demand, energy_cost, start_cost
        model = build_model(*args)
        result, elapsed = solve_milp(model, f"{name} MILP")
        require_status(result, 0)
        runs.append((name, STATUS[result.status], result.fun, elapsed, result.mip_gap,
                     result.mip_dual_bound, result.mip_node_count, result.x, args, 1, True))
        if name != "base":
            continue
        lp = relaxation(model, "base LP relaxation")
        loose_model = build_model(*args, multiplier=1000)
        loose_lp = relaxation(loose_model, "base LP relaxation, loose M")
        loose, loose_time = solve_milp(loose_model, "base MILP, loose M")
        for solved in [lp, loose_lp, loose]:
            require_status(solved, 0)
        start = perf_counter()
        heuristic = greedy(*args)
        greedy_time = perf_counter() - start
        runs.extend([
            ("base_lp", "optimal", lp.fun, None, None, None, None, lp.x, args, 1, False),
            ("loose_lp", "optimal", loose_lp.fun, None, None, None, None, loose_lp.x, args, 1000, False),
            ("loose_milp", "optimal", loose.fun, loose_time, loose.mip_gap,
             loose.mip_dual_bound, loose.mip_node_count, loose.x, args, 1000, True),
            ("greedy", "feasible", cost_of(heuristic, energy_cost, start_cost), greedy_time,
             None, None, None, heuristic.ravel(), args, 1, True),
        ])
    summary = []
    for name, status, objective, elapsed, gap, bound, nodes, x, args, factor, integral in runs:
        available, demand = args[1:3]
        plan = x.reshape(3, *available.shape)
        checks[name] = verify(plan, *args, objective, multiplier=factor, integral=integral)
        summary.append([name, status, objective, elapsed, gap, bound, nodes,
                        float(np.sum(plan[0] * energy_cost)), float(np.sum(plan[2] * start_cost))])
        for t, g in np.ndindex(available.shape):
            plan_rows.append([name, t + 1, data["ids"][g], demand[t], available[t], *plan[:, t, g]])
    write_csv(RESULTS, "scenarios", ["scenario", "status", "cost", "seconds", "mip_gap_fraction",
                                    "dual_bound", "nodes", "energy_cost", "startup_cost"], summary)
    write_csv(RESULTS, "solution", ["scenario", "hour", "generator", "demand", "available", "p", "y", "v"],
              plan_rows)
    write_check(RESULTS, "checks", checks)
    return draw_benchmark(benchmark(data))


figure = run_with_log(run_lab, data, RESULTS)

## Результаты

Планы, проверки и лог сохранены в папке `results`.

In [ ]:
show_table(RESULTS / "scenarios.csv")
show_table(RESULTS / "scaling.csv")
with BytesIO() as buffer:
    figure.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
display(JSON(json.loads((RESULTS / "checks.json").read_text(encoding="utf-8"))))